# MuseTalk 1.5 — GPU Colab POC

Runs **MuseTalk** on a Colab **GPU** (near real-time), then reports **timing** and **quality metrics** to compare with the other notebooks.

> Set **Runtime → Change runtime type → GPU** first.

## Step 0 — confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1 — clone + install (CUDA PyTorch + mmlab)

In [ ]:
%cd /content
!git clone https://github.com/TMElyralab/MuseTalk.git
%cd /content/MuseTalk
!apt-get -qq install -y ffmpeg > /dev/null
# Pin torch 2.1 + cu121 so the mmlab GPU wheels have a matching build. mmcv is
# very picky about the torch/CUDA version - this is the usual MuseTalk-on-Colab snag.
!pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

# Inference deps (skip tensorflow/gradio - unused); numpy pinned for compatibility.
!pip -q install "numpy==1.23.5" diffusers==0.30.2 accelerate==0.28.0 transformers==4.39.2 \
  huggingface_hub==0.30.2 librosa==0.11.0 einops==0.8.1 omegaconf ffmpeg-python moviepy \
  soundfile==0.12.1 opencv-python==4.9.0.80 requests imageio imageio-ffmpeg gdown

# mmlab stack: GPU mmcv wheel matching torch 2.1 / cu121 (do NOT use `mim install mmcv`
# unpinned - it may grab a build that doesn't match Colab's torch).
!pip -q install -U openmim "setuptools==69.5.1" wheel
!pip -q install mmengine "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html
!pip -q install chumpy==0.70 --no-build-isolation
!pip -q install "mmdet==3.2.0" "mmpose==1.3.1" --no-build-isolation
!python -c "import mmpose, mmcv, torch; print('mmpose', mmpose.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## Step 2 — download weights (~4 GB)

In [ ]:
import urllib.request, os
from huggingface_hub import hf_hub_download
M='models'
hf=[('TMElyralab/MuseTalk','musetalkV15/musetalk.json',M),
    ('TMElyralab/MuseTalk','musetalkV15/unet.pth',M),
    ('stabilityai/sd-vae-ft-mse','config.json',M+'/sd-vae'),
    ('stabilityai/sd-vae-ft-mse','diffusion_pytorch_model.bin',M+'/sd-vae'),
    ('openai/whisper-tiny','config.json',M+'/whisper'),
    ('openai/whisper-tiny','pytorch_model.bin',M+'/whisper'),
    ('openai/whisper-tiny','preprocessor_config.json',M+'/whisper'),
    ('yzd-v/DWPose','dw-ll_ucoco_384.pth',M+'/dwpose')]
for r,f,o in hf: hf_hub_download(repo_id=r, filename=f, local_dir=o)
os.makedirs(M+'/face-parse-bisent', exist_ok=True)
urllib.request.urlretrieve('https://download.pytorch.org/models/resnet18-5c106cde.pth',
    M+'/face-parse-bisent/resnet18-5c106cde.pth')
!gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth
print('weights ready')

## Step 3 — inputs
MuseTalk accepts an **image or a video** + audio.

In [ ]:
from google.colab import files
up = files.upload()
names = list(up.keys())
VIDEO = [n for n in names if n.lower().endswith(('.png','.jpg','.jpeg','.mp4','.mov'))][0]
AUDIO = [n for n in names if n.lower().endswith(('.wav','.mp3','.m4a'))][0]
import os; os.makedirs('data', exist_ok=True)
open('configs/inference/poc.yaml','w').write(
    f'task_0:\n video_path: "{VIDEO}"\n audio_path: "{AUDIO}"\n')
print('config written for', VIDEO, AUDIO)

## Step 4 — run MuseTalk (GPU) + timing
(For a still image the repo logs a harmless `save_dir_full` error *after* writing the mp4.)

In [ ]:
import time; _t = time.time()
!python -m scripts.inference \
  --inference_config configs/inference/poc.yaml --result_dir results/poc \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json --version v15 || true
print(f'[TIME] MuseTalk end-to-end: {time.time()-_t:.1f}s')
import glob; OUT = glob.glob('results/poc/v15/*_*.mp4')[0]; print('output:', OUT)

## Step 5 — quality metrics

In [ ]:
# Quality metrics (reference-free): CSIM identity + mouth/face sharpness.
# Same definitions as the local wav2lip/evaluate.py, so numbers are comparable
# across notebooks. Needs OpenCV YuNet + SFace (downloaded here if missing).
import urllib.request, os, cv2, numpy as np
_M = {
 "yunet.onnx": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
 "sface.onnx": "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
}
for _n, _u in _M.items():
    if not os.path.exists(_n):
        urllib.request.urlretrieve(_u, _n)
_det = cv2.FaceDetectorYN.create("yunet.onnx", "", (320, 320), 0.6, 0.3, 5000)
_rec = cv2.FaceRecognizerSF.create("sface.onnx", "")

def _big(f):
    h, w = f.shape[:2]; _det.setInputSize((w, h)); _, fc = _det.detect(f)
    return None if fc is None or len(fc) == 0 else max(fc, key=lambda z: z[-1])

def _lap(g):
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

def _mouth(fr, face):
    lm = face[4:14].reshape(5, 2); (rx, ry), (lx, ly) = lm[3], lm[4]
    cx, cy = int((rx + lx) / 2), int((ry + ly) / 2)
    mw = int(abs(lx - rx) * 1.6) or 40; mh = int(mw * 0.7)
    c = fr[max(0, cy - mh // 2):cy + mh // 2, max(0, cx - mw // 2):cx + mw // 2]
    return 0.0 if c.size == 0 else _lap(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY))

def _first(path):
    if path.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp")):
        return cv2.imread(path)
    cap = cv2.VideoCapture(path); ok, fr = cap.read(); cap.release(); return fr

def metrics(video, source, every=5):
    src = _first(source); sf = _big(src)
    sfeat = _rec.feature(_rec.alignCrop(src, sf))
    cap = cv2.VideoCapture(video); cs, ms, fs = [], [], []; i = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if i % every == 0:
            f = _big(fr)
            if f is not None:
                cs.append(float(_rec.match(sfeat, _rec.feature(_rec.alignCrop(fr, f)),
                                           cv2.FaceRecognizerSF_FR_COSINE)))
                ms.append(_mouth(fr, f)); x, y, bw, bh = f[:4].astype(int)
                roi = fr[max(0, y):y + bh, max(0, x):x + bw]
                if roi.size:
                    fs.append(_lap(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)))
        i += 1
    cap.release()
    _m = lambda a: float(np.mean(a)) if a else 0.0
    print(f"CSIM(identity)={_m(cs):.3f}   mouth_sharp={_m(ms):.1f}   "
          f"face_sharp={_m(fs):.1f}   frames={len(cs)}   [CSIM>0.6=same person]")


In [ ]:
print('== MuseTalk =='); metrics(OUT, VIDEO)

## Step 6 — preview

In [ ]:
from IPython.display import HTML
from base64 import b64encode
d = b64encode(open(OUT,'rb').read()).decode()
HTML(f'<video width=420 controls><source src="data:video/mp4;base64,{d}" type="video/mp4"></video>')